In [1]:
from google.colab import auth
auth.authenticate_user(project_id="ydf-distributed-spo-gbt")

In [9]:
!gcloud services enable compute.googleapis.com
!gcloud services enable cloudbuild.googleapis.com
!gcloud services enable aiplatform.googleapis.com

Operation "operations/acf.p2-491585234199-1e2f9a5d-a835-405c-8398-3814a68aa9ad" finished successfully.


In [5]:
!gcloud projects describe ydf-distributed-spo-gbt --format="value(projectNumber)"

491585234199


In [ ]:
# docker packing and job running permissions
%%bash
# Sleep for a few seconds to let GCP propagate the newly created service account
sleep 10

gcloud projects add-iam-policy-binding ydf-distributed-spo-gbt \
    --member="serviceAccount:491585234199-compute@developer.gserviceaccount.com" \
    --role="roles/storage.objectViewer"

gcloud projects add-iam-policy-binding ydf-distributed-spo-gbt \
    --member="serviceAccount:491585234199-compute@developer.gserviceaccount.com" \
    --role="roles/run.builder"

gcloud projects add-iam-policy-binding ydf-distributed-spo-gbt \
    --member="serviceAccount:491585234199-compute@developer.gserviceaccount.com" \
    --role="roles/artifactregistry.createOnPushWriter"

### Using Adult dataset

Later, use HIGGS, and rest of our natural datasets

In [12]:
import pandas as pd

ds_path = "https://raw.githubusercontent.com/google/yggdrasil-decision-forests/main/yggdrasil_decision_forests/test_data/dataset"
train_ds = pd.read_csv(f"{ds_path}/adult_train.csv")
test_ds = pd.read_csv(f"{ds_path}/adult_test.csv")

print("The dataset has",len(train_ds),"training examples")

The dataset has 22792 training examples


In [15]:
# using ariel_ydf_bucket
# Shard dataset into 10
import os

def split_dataset(
    dataset: pd.DataFrame, tmp_dir: str, num_shards: int
) -> list[str]:
  """Splits a csv file into multiple csv files."""

  os.makedirs(tmp_dir,exist_ok=True)
  num_row_per_shard = (dataset.shape[0] + num_shards - 1) // num_shards
  paths = []
  for shard_idx in range(num_shards):
    begin_idx = shard_idx * num_row_per_shard
    end_idx = (shard_idx + 1) * num_row_per_shard
    shard_dataset = dataset.iloc[begin_idx:end_idx]
    shard_path = os.path.join(tmp_dir , f"shard_{shard_idx}.csv")
    paths.append(shard_path)
    shard_dataset.to_csv(shard_path, index=False)
  return paths

split_dataset(train_ds, "gs://ariel_ydf_bucket/train_dataset", 10)

['gs://ariel_ydf_bucket/train_dataset/shard_0.csv',
 'gs://ariel_ydf_bucket/train_dataset/shard_1.csv',
 'gs://ariel_ydf_bucket/train_dataset/shard_2.csv',
 'gs://ariel_ydf_bucket/train_dataset/shard_3.csv',
 'gs://ariel_ydf_bucket/train_dataset/shard_4.csv',
 'gs://ariel_ydf_bucket/train_dataset/shard_5.csv',
 'gs://ariel_ydf_bucket/train_dataset/shard_6.csv',
 'gs://ariel_ydf_bucket/train_dataset/shard_7.csv',
 'gs://ariel_ydf_bucket/train_dataset/shard_8.csv',
 'gs://ariel_ydf_bucket/train_dataset/shard_9.csv']

In [17]:
# test data
split_dataset(test_ds, "gs://ariel_ydf_bucket/valid_dataset", 10)

['gs://ariel_ydf_bucket/valid_dataset/shard_0.csv',
 'gs://ariel_ydf_bucket/valid_dataset/shard_1.csv',
 'gs://ariel_ydf_bucket/valid_dataset/shard_2.csv',
 'gs://ariel_ydf_bucket/valid_dataset/shard_3.csv',
 'gs://ariel_ydf_bucket/valid_dataset/shard_4.csv',
 'gs://ariel_ydf_bucket/valid_dataset/shard_5.csv',
 'gs://ariel_ydf_bucket/valid_dataset/shard_6.csv',
 'gs://ariel_ydf_bucket/valid_dataset/shard_7.csv',
 'gs://ariel_ydf_bucket/valid_dataset/shard_8.csv',
 'gs://ariel_ydf_bucket/valid_dataset/shard_9.csv']

## Create Docker for VertexAI

In [18]:
%%writefile train.py
import argparse
import dataclasses
import json
import os
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union
import ydf

parser = argparse.ArgumentParser()
# Path to training dataset. Should be prefixed with the dataset type e.g. 'csv':.
# See the supported formats at https://ydf.readthedocs.io/en/latest/dataset_formats/
parser.add_argument("--train_ds", type=str, required=True)
# Path to validation dataset. If empty, the model is trained without validation.
parser.add_argument("--valid_ds", type=str)
# Path to test dataset. If empty, the model is not evaluated.
parser.add_argument("--test_ds", type=str)
# Path to save the model.
parser.add_argument("--model", type=str, required=True)
# Work directory containing the temporary working data. Only used for distributed training.
parser.add_argument("--work_dir", default="", type=str)
# Label column to predict.
parser.add_argument("--label", type=str, required=True)
# Is the training distributed, or on a single machine?
parser.add_argument("--distributed", action="store_true")
args = parser.parse_args()


def main():
  print("Arguments:\n", args)

  if args.distributed:
    main_distributed()
  else:
    main_in_process()


def main_in_process():
  ydf.verbose(2)

  print("Train model in process on", args.train_ds)
  learner = ydf.GradientBoostedTreesLearner(label=args.label)
  model = learner.train(args.train_ds, valid=args.valid_ds)

  print("Save model in", args.model)
  model.save(args.model)

  if args.test_ds:
    print("Evaluate model on", args.test_ds)
    evaluation = model.evaluate(args.test_ds)
    print(evaluation)


def main_distributed():
  ydf.verbose(2)

  # Gather the manager and workers configuration.
  cluster_config = ydf.util.get_vertex_ai_cluster_spec()
  print("cluster_config:\n", cluster_config)

  if cluster_config.is_worker:
    # This machine is running a worker.
    ydf.start_worker(cluster_config.port)
    return

  print("Train model with distribution on", args.train_ds)
  learner = ydf.DistributedGradientBoostedTreesLearner(
      label=args.label,
      workers=cluster_config.workers,
      working_dir=args.work_dir,
      resume_training=True,
  )
  model = learner.train(args.train_ds, valid=args.valid_ds)

  print("Save model in", args.model)
  model.save(args.model)

  if args.test_ds:
    print("Evaluate model on", args.test_ds)
    evaluation = model.evaluate(args.test_ds)
    print(evaluation)


if __name__ == "__main__":
  main()

Writing train.py


In [22]:
# Create Dockerfile for VertexAI
%%writefile Dockerfile
FROM python:3.12
WORKDIR /root

RUN apt-get update && apt-get -y install sudo

# Install YDF from Pip
RUN python3 -m pip install --no-cache-dir ydf

RUN echo '[GoogleCompute]\nservice_account = default' > /etc/boto.cfg

COPY train.py /root/train.py

ENTRYPOINT ["python3", "train.py"]

Overwriting Dockerfile


In [40]:
# Compile Docker & upload to GC
%%bash
gcloud builds submit --tag us-east1-docker.pkg.dev/ydf-distributed-spo-gbt/ydf-repo/train-ydf

----------------------------- REMOTE BUILD OUTPUT ------------------------------
starting build "260528d9-6860-4c8f-9878-c17f779173b5"

FETCHSOURCE
Fetching storage object: gs://ydf-distributed-spo-gbt_cloudbuild/source/1791770412.426682-4ee7a85cd6964e3db0f9d4ef0e78c24e.tgz#1791770476692556
Copying gs://ydf-distributed-spo-gbt_cloudbuild/source/1791770412.426682-4ee7a85cd6964e3db0f9d4ef0e78c24e.tgz#1791770476692556...
\ [1 files][  1.0 GiB/  1.0 GiB]   59.0 MiB/s                                   
Operation completed over 1 objects/1.0 GiB.
BUILD
Already have image (with digest): gcr.io/cloud-builders/gcb-internal
DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            Install the buildx component to build images with BuildKit:
            https://docs.docker.com/go/buildx/

Step 1/7 : FROM python:3.12
3.12: Pulling from library/python
4846f2900730: Pulling fs layer
62fe2e747f69: Pulling fs layer
4dc3950125eb: Pulling fs layer
b2eb680928d4: Pul

Creating temporary archive of 67 file(s) totalling 1.1 GiB before compression.
Uploading tarball of [.] to [gs://ydf-distributed-spo-gbt_cloudbuild/source/1791770412.426682-4ee7a85cd6964e3db0f9d4ef0e78c24e.tgz]
Created [https://cloudbuild.googleapis.com/v1/projects/ydf-distributed-spo-gbt/locations/global/builds/260528d9-6860-4c8f-9878-c17f779173b5].
Logs are available at [ https://console.cloud.google.com/cloud-build/builds/260528d9-6860-4c8f-9878-c17f779173b5?project=491585234199 ].
Waiting for build to complete. Polling interval: 1 second(s).


Vertex AI Custom Jobs doesn't have access to all GCP instance types

E.g. N4d AMD-based instances aren't supported

## Start the job

In [43]:
%%bash
# Manager and Workers Configuration
gcloud ai custom-jobs create \
    --region=us-east1 \
    --project=ydf-distributed-spo-gbt \
    --worker-pool-spec=replica-count=1,machine-type='n2-highmem-2',container-image-uri='us-east1-docker.pkg.dev/ydf-distributed-spo-gbt/ydf-repo/train-ydf' \
    --worker-pool-spec=replica-count=5,machine-type='n2-highmem-2',container-image-uri='us-east1-docker.pkg.dev/ydf-distributed-spo-gbt/ydf-repo/train-ydf' \
    --display-name=train-ydf-job \
--args=\
--train_ds=csv:gs://ariel_ydf_bucket/train_dataset/shard_*.csv,\
--valid_ds=csv:gs://ariel_ydf_bucket/valid_dataset/shard_*.csv,\
--model=gs://ariel_ydf_bucket/model,\
--work_dir=gs://ariel_ydf_bucket/work_dir,\
--distributed,\
--label=income

Using endpoint [https://us-east1-aiplatform.googleapis.com/]
CustomJob [projects/491585234199/locations/us-east1/customJobs/5292760010713989120] is submitted successfully.

Your job is still active. You may view the status of your job with the command

  $ gcloud ai custom-jobs describe projects/491585234199/locations/us-east1/customJobs/5292760010713989120

or continue streaming the logs with the command

  $ gcloud ai custom-jobs stream-logs projects/491585234199/locations/us-east1/customJobs/5292760010713989120


### Get Job status - Runtime

In [44]:
%%bash
sleep 5

In [48]:
# check job status
from google.cloud import aiplatform

# Initialize the AI Platform SDK
aiplatform.init(project="ydf-distributed-spo-gbt", location="us-east1")

# Automatically retrieve the most recently created custom job
jobs = aiplatform.CustomJob.list(order_by="create_time desc")

if not jobs:
    print("No custom jobs found.")
else:
    custom_job = jobs[0]
    job_id = custom_job.name

    print(f"=== Job Report for {custom_job.display_name} (ID: {job_id}) ===")
    print(f"State: {custom_job.state.name}")
    print(f"Start Time: {custom_job.start_time}")
    print(f"End Time: {custom_job.end_time}")

    if custom_job.error:
        print("\n=== Failure Details ===")
        print(f"Error Code: {custom_job.error.code}")
        print(f"Error Message: {custom_job.error.message}")
    else:
        print("\nJob completed successfully without errors (or is still running).")

=== Job Report for train-ydf-job (ID: 5292760010713989120) ===
State: JOB_STATE_SUCCEEDED
Start Time: 2026-10-12 02:14:22+00:00
End Time: 2026-10-12 02:14:52+00:00

=== Failure Details ===
Error Code: 0
Error Message: 


### Accuracy

Your `train.py` script saves the trained YDF model to `gs://ariel_ydf_bucket/model`. To inspect the final accuracy, you can download the evaluation results directly from the saved model folder or load the model in this notebook environment using `ydf` (if installed locally):

In [49]:
print("=== Checking active gcloud account and project ===")
!gcloud config get-value project

print("\n=== Listing Artifact Registry repositories ===")
!gcloud artifacts repositories list || echo "Failed to list registries"

print("\n=== Checking built images in the project ===")
!gcloud container images list || echo "No images found in GCR"

=== Checking active gcloud account and project ===
ydf-distributed-spo-gbt

=== Listing Artifact Registry repositories ===
Listing items under project ydf-distributed-spo-gbt, across all locations.

                                                                                ARTIFACT_REGISTRY
REPOSITORY  FORMAT  MODE                 DESCRIPTION                                     LOCATION  LABELS  ENCRYPTION          CREATE_TIME          UPDATE_TIME          SIZE (MB)
ydf-repo    DOCKER  STANDARD_REPOSITORY  Docker repository for YDF distributed training  us-east1          Google-managed key  2026-10-12T01:46:44  2026-10-12T02:04:13  450.711

=== Checking built images in the project ===
ERROR: (gcloud.container.images.list) Bad status during token exchange: 404
b'{"errors":[{"code":"NAME_UNKNOWN","message":"Repository \\"gcr.io\\" not found"}]}\n'
No images found in GCR


In [35]:
import ydf

# Directly load and describe the model from GCS
model = ydf.load_model("gs://ariel_ydf_bucket/model")
model.describe()

Loading model from gs://ariel_ydf_bucket/model


RuntimeError: FAILED_PRECONDITION: File prefix cannot be autodetected: 0 models exist in directory "gs://ariel_ydf_bucket/model"

In [39]:
%%bash
# Enable the Artifact Registry API
gcloud services enable artifactregistry.googleapis.com

# Create a new Docker repository in us-east1
gcloud artifacts repositories create ydf-repo \
    --repository-format=docker \
    --location=us-east1 \
    --description="Docker repository for YDF distributed training"

# Verify creation
gcloud artifacts repositories list --location=us-east1

                                                                                ARTIFACT_REGISTRY
REPOSITORY  FORMAT  MODE                 DESCRIPTION                                     LOCATION  LABELS  ENCRYPTION          CREATE_TIME          UPDATE_TIME          SIZE (MB)
ydf-repo    DOCKER  STANDARD_REPOSITORY  Docker repository for YDF distributed training  us-east1          Google-managed key  2026-10-12T01:46:44  2026-10-12T01:46:44  0


Create request issued for: [ydf-repo]
Waiting for operation [projects/ydf-distributed-spo-gbt/locations/us-east1/operations/49977df7-b0ef-4f02-821d-49c1cdf64971] to complete...
...................................................................done.
Created repository [ydf-repo].
Listing items under project ydf-distributed-spo-gbt, location us-east1.

